In [1]:
!pip install imbalance-metrics smogn

In [2]:
!pip install ImbalancedLearningRegression


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.2/77.2 kB 5.8 MB/s eta 0:00:00


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import PchipInterpolator
from smogn import phi, phi_ctrl_pts
from sklearn.model_selection import KFold
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
from scipy.spatial import distance
from scipy.spatial import distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor,
    BaggingRegressor
)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.linear_model import (
    ARDRegression,
    SGDRegressor,
    PassiveAggressiveRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from smogn import smoter
import ImbalancedLearningRegression as iblr

In [4]:
models_pool = [
    SVR(),
    RandomForestRegressor(n_estimators=100, random_state=42),
    DecisionTreeRegressor(random_state=42),
    MLPRegressor(hidden_layer_sizes=(50,50), max_iter=1000, random_state=42),
    BaggingRegressor(random_state=42),
    XGBRegressor(n_estimators=100, random_state=42)
]

# models_pool = [
#     RandomForestRegressor(n_estimators=100, random_state=42),
#     ExtraTreesRegressor(n_estimators=100, random_state=42),
#     XGBRegressor(n_estimators=100, random_state=42)
# ]

In [5]:
def instance_hardness_regression(X, y, models=models_pool, distance='l1', n_splits=10, logs=False, gamma_fixo = False, fator_divisao_gamma = 1):
    n = len(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    if gamma_fixo:
      gamma = 1
    else:
      ### Justamente o que diz no artigo, que é a media do target elevado a 2
      gamma = np.mean(y**2) / fator_divisao_gamma #### Gamma menor -> Maior sensibilidade a erros, deixando o valor final de IG mais proximo de 1 ( Por padrão é np.mean(y**2))

    preds_pool = np.zeros((n, len(models)))

    ### Cross-validation ###
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        if logs:
          print(f"Iniciando fold {fold_idx}/{n_splits}...")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train = y[train_idx]

        ### ADICIONAR OS TIPOS DE BALANCEAMENTO NO X_TRAIN AQUI
        for j, model in enumerate(models):
            if logs:
              print(f"Treinando modelo {j+1}/{len(models)}: {type(model).__name__} ...")
            clone_model = type(model)(**model.get_params())
            clone_model.fit(X_train, y_train)
            preds_pool[test_idx, j] = clone_model.predict(X_test)

    if distance == 'l1':
        dists = np.abs(y.reshape(-1, 1) - preds_pool)
    elif distance == 'l2':
        dists = (y.reshape(-1, 1) - preds_pool)**2
    else:
        raise ValueError("Use 'l1' ou 'l2'.")

    exp_term = np.exp(-dists / gamma)
    ih_values = 1 - np.mean(exp_term, axis=1) ### Isso equivale a divisão por |L|

    return ih_values

In [6]:
## load dependencies - third party
import numpy as np
import pandas as pd
import random as rd

## generate synthetic observations
def over_sampling_gn(

    ## arguments / inputs
    data,       ## training set
    index,      ## index of input data
    perc,       ## over / under sampling
    pert,       ## perturbation / noise percentage
    replace,    ## sampling replacement (bool)

    ):

    """
    generates synthetic observations and is the primary function underlying the
    over-sampling technique utilized in the higher main function 'gn()', the
    4 step procedure for generating synthetic observations is:

    1) pre-processing: temporarily removes features without variation, label
    encodes nominal / categorical features, and subsets the training set into
    two data sets by data type: numeric / continuous, and nominal / categorical

    2) over-sampling: GN, which perturb the interpolated values with gaussian noise

    3) post processing: restores original values for label encoded features,
    reintroduces constant features previously removed, converts any interpolated
    negative values to zero in the case of non-negative features

    returns a pandas dataframe containing synthetic observations of the training
    set which are then returned to the higher main function 'gn()'

    ref:

    Branco, P., Torgo, L., Ribeiro, R. (2017).
    SMOGN: A Pre-Processing Approach for Imbalanced Regression.
    Proceedings of Machine Learning Research, 74:36-50.
    http://proceedings.mlr.press/v74/branco17a/branco17a.pdf.

    Branco, P., Ribeiro, R., Torgo, L. (2017).
    Package 'UBL'. The Comprehensive R Archive Network (CRAN).
    https://cran.r-project.org/web/packages/UBL/UBL.pdf.

    Branco, P., Torgo, L., & Ribeiro, R. P. (2019).
    Pre-processing approaches for imbalanced distributions in regression.
    Neurocomputing, 343, 76-99.
    https://www.sciencedirect.com/science/article/abs/pii/S0925231219301638

    Kunz, N., (2019). SMOGN.
    https://github.com/nickkunz/smogn
    """

    ## subset original dataframe by bump classification index
    data = data.iloc[index]

    ## store dimensions of data subset
    n = len(data)
    d = len(data.columns)

    ## store original data types
    feat_dtypes_orig = [None] * d

    for j in range(d):
        feat_dtypes_orig[j] = data.iloc[:, j].dtype

    ## find non-negative numeric features
    feat_non_neg = []
    num_dtypes = ["int64", "float64"]

    for j in range(d):
        if data.iloc[:, j].dtype in num_dtypes and any(data.iloc[:, j] > 0):
            feat_non_neg.append(j)

    ## find features without variation (constant features)
    feat_const = data.columns[data.nunique() == 1]

    ## temporarily remove constant features
    if len(feat_const) > 0:

        ## create copy of orignal data and omit constant features
        data_orig = data.copy()
        data = data.drop(data.columns[feat_const], axis = 1)

        ## store list of features with variation
        feat_var = list(data.columns.values)

        ## reindex features with variation
        for i in range(d - len(feat_const)):
            data.rename(columns = {
                data.columns[i]: i
                }, inplace = True)

        ## store new dimension of feature space
        d = len(data.columns)

    ## create copy of data containing variation
    data_var = data.copy()

    ## create global feature list by column index
    feat_list = list(data.columns.values)

    ## create nominal feature list and
    ## label encode nominal / categorical features
    ## (strictly label encode, not one hot encode)
    feat_list_nom = []
    nom_dtypes = ["object", "bool", "datetime64"]

    # Unknown warning, may be handled later
    pd.options.mode.chained_assignment = None

    for j in range(d):
        if data.dtypes[j] in nom_dtypes:
            feat_list_nom.append(j)
            data.iloc[:, j] = pd.Categorical(pd.factorize(
                data.iloc[:, j])[0])

    data = data.apply(pd.to_numeric)

    ## create numeric feature list
    feat_list_num = list(set(feat_list) - set(feat_list_nom))

    ## calculate ranges for numeric / continuous features
    ## (includes label encoded features)
    feat_ranges = list(np.repeat(1, d))

    if len(feat_list_nom) > 0:
        for j in feat_list_num:
            feat_ranges[j] = max(data.iloc[:, j]) - min(data.iloc[:, j])
    else:
        for j in range(d):
            feat_ranges[j] = max(data.iloc[:, j]) - min(data.iloc[:, j])

    ## subset feature ranges to include only numeric features
    ## (excludes label encoded features)
    feat_ranges_num = [feat_ranges[i] for i in feat_list_num]

    ## subset data by either numeric / continuous or nominal / categorical
    data_num = data.iloc[:, feat_list_num]
    data_nom = data.iloc[:, feat_list_nom]

    ## get number of features for each data type
    feat_count_num = len(feat_list_num)
    feat_count_nom = len(feat_list_nom)


    ## number of new synthetic observations for each rare observation
    x_synth = int(perc - 1)

    ## total number of new synthetic observations to generate
    n_synth = int(n * (perc - 1 - x_synth))

    ## randomly index data by the number of new synthetic observations
    r_index = np.random.choice(
        a = tuple(range(0, n)),
        size = n_synth,
        replace = replace,
        p = None
    )

    ## create null matrix to store new synthetic observations
    synth_matrix = np.ndarray(shape = ((x_synth * n + n_synth), d))

    # modified
    if x_synth > 0:
        for i in range(n):
            for j in range(x_synth):
                for attr in range(d):
                    if pd.isna(data.iloc[i, attr]):
                            synth_matrix[i * x_synth + j, attr] = None
                    if attr in feat_list_nom:
                        synth_matrix[i * x_synth + j, attr] = np.random.choice(
                            a=list(data.iloc[:, attr]))
                    else:
                        synth_matrix[i * x_synth + j, attr] = data.iloc[
                            i, attr] + np.random.normal(0,
                            np.sqrt(pert * np.std(list(data.iloc[:, attr]))))

    # modified
    if n_synth > 0:
        count = 0
        for i in r_index:
            for attr in range(d):
                    if pd.isna(data.iloc[i, attr]):
                            synth_matrix[x_synth * n + count, attr] = None
                    if attr in feat_list_nom:
                        synth_matrix[x_synth * n + count, attr] = np.random.choice(
                            a=list(data.iloc[:, attr]))
                    else:
                        synth_matrix[x_synth * n + count, attr] = data.iloc[
                            i, attr] + np.random.normal(0,
                            np.sqrt(pert * np.std(list(data.iloc[:, attr]))))
            ## close loop counter
            count = count + 1

    ## convert synthetic matrix to dataframe
    data_new = pd.DataFrame(synth_matrix)

    ## synthetic data quality check
    if sum(data_new.isnull().sum()) > 0:
        raise ValueError("oops! synthetic data contains missing values")

    ## replace label encoded values with original values
    for j in feat_list_nom:
        code_list = data.iloc[:, j].unique()
        cat_list = data_var.iloc[:, j].unique()

        for x in code_list:
            data_new.iloc[:, j] = data_new.iloc[:, j].replace(x, cat_list[x])

    ## reintroduce constant features previously removed
    if len(feat_const) > 0:
        data_new.columns = feat_var

        for j in range(len(feat_const)):
            data_new.insert(
                loc = int(feat_const[j]),
                column = feat_const[j],
                value = np.repeat(
                    data_orig.iloc[0, feat_const[j]],
                    len(synth_matrix))
            )

    ## convert negative values to zero in non-negative features
    for j in feat_non_neg:
        # data_new.iloc[:, j][data_new.iloc[:, j] < 0] = 0
        data_new.iloc[:, j] = data_new.iloc[:, j].clip(lower = 0)

    ## return over-sampling results dataframe
    return data_new

In [7]:
def apply_gn_ih(

    ## main arguments / inputs
    data,                     ## training set (pandas dataframe)
    y,                        ## response variable y by name (string)
    pert = 0.02,              ## perturbation / noise percentage (pos real)
    samp_method = "balance",  ## over / under sampling ("balance" or extreme")
    under_samp = True,        ## under sampling (bool)
    drop_na_col = True,       ## auto drop columns with nan's (bool)
    drop_na_row = True,       ## auto drop rows with nan's (bool)
    replace = False,          ## sampling replacement (bool)
    manual_perc = False,      ## user defines percentage of under-sampling and over-sampling  # added
    perc_u = -1,              ## percentage of under-sampling  # added
    perc_o = -1,              ## percentage of over-sampling  # added

    ## phi relevance function arguments / inputs
    rel_thres = 0.5,          ## relevance threshold considered rare (pos real)
    rel_method = "auto",      ## relevance method ("auto" or "manual")
    rel_xtrm_type = "both",   ## distribution focus ("high", "low", "both")
    rel_coef = 1.5,           ## coefficient for box plot (pos real)
    rel_ctrl_pts_rg = None    ## input for "manual" rel method  (2d array)

    ):

    """
    the main function, designed to help solve the problem of imbalanced data
    for regression; GN applies the combintation of under-sampling the majority
    class (in the case of regression, values commonly found near the mean of
    a normal distribution in the response variable y) and over-sampling the
    minority class (rare values in a normal distribution of y, typically found
    at the tails)

    procedure begins with a series of pre-processing steps, and to ensure no
    missing values (nan's), sorts the values in the response variable y by
    ascending order, and fits a function 'phi' to y, corresponding phi values
    (between 0 and 1) are generated for each value in y, the phi values are
    then used to determine if an observation is either normal or rare by the
    threshold specified in the argument 'rel_thres'

    normal observations are placed into a majority class subset (normal bin)
    and are under-sampled, while rare observations are placed in a seperate
    minority class subset (rare bin) where they're over-sampled

    under-sampling is applied by a random sampling from the normal bin based
    on a calculated percentage control by the argument 'samp_method', if the
    specified input of 'samp_method' is "balance", less under-sampling (and
    over-sampling) is conducted, and if "extreme" is specified more under-
    sampling (and over-sampling is conducted)

    over-sampling is applied by GN, which perturb the interpolated values
    with gaussian noise

    GN is only applied to numeric / continuous features, synthetic values
    found in nominal / categorical features, are generated by randomly
    selecting observed values found within their respective feature

    procedure concludes by post-processing and returns a modified pandas data
    frame containing under-sampled and over-sampled (synthetic) observations,
    the distribution of the response variable y should more appropriately
    reflect the minority class areas of interest in y that are under-
    represented in the original training set

    note that users can also decide the percentage of under-sampling and
    over-sampling on each bin by setting manual_perc to True and assigning
    values to perc_u and perc_o, these values will directly replace the
    calculated percentage of each bin

    ref:

    Branco, P., Torgo, L., Ribeiro, R. (2017).
    SMOGN: A Pre-Processing Approach for Imbalanced Regression.
    Proceedings of Machine Learning Research, 74:36-50.
    http://proceedings.mlr.press/v74/branco17a/branco17a.pdf.

    Branco, P., Torgo, L., & Ribeiro, R. P. (2019).
    Pre-processing approaches for imbalanced distributions in regression.
    Neurocomputing, 343, 76-99.
    https://www.sciencedirect.com/science/article/abs/pii/S0925231219301638

    Kunz, N., (2019). SMOGN.
    https://github.com/nickkunz/smogn
    """

    ## pre-process missing values
    if bool(drop_na_col) == True:
        data = data.dropna(axis = 1)  ## drop columns with nan's

    if bool(drop_na_row) == True:
        data = data.dropna(axis = 0)  ## drop rows with nan's

    ## quality check for missing values in dataframe
    if data.isnull().values.any():
        raise ValueError("cannot proceed: data cannot contain NaN values")

    ## quality check for y
    if isinstance(y, str) is False:
        raise ValueError("cannot proceed: y must be a string")

    if y in data.columns.values is False:
        raise ValueError("cannot proceed: y must be an header name (string) \
               found in the dataframe")

    ## quality check for perturbation
    if pert > 1 or pert <= 0:
        raise ValueError("pert must be a real number number: 0 < R < 1")

    ## quality check for sampling method
    if samp_method in ["balance", "extreme"] is False:
        raise ValueError("samp_method must be either: 'balance' or 'extreme'")

    # added
    ## quality check for sampling percentage
    if manual_perc:
        if perc_u == -1 or perc_o == -1:
            raise ValueError("cannot proceed: require percentage of under-sampling and over-sampling if manual_perc == True")
        if perc_u >= 1 or perc_u <= 0:
            raise ValueError("percentage of under-sampling must be a real number between 0 and 1")
        if perc_o <= 0:
            raise ValueError("percentage of over-sampling must be a positve real number")

    ## quality check for relevance threshold parameter
    if rel_thres == None:
        raise ValueError("cannot proceed: relevance threshold required")

    if rel_thres > 1 or rel_thres <= 0:
        raise ValueError("rel_thres must be a real number number: 0 < R < 1")

    ## store data dimensions
    n = len(data)
    d = len(data.columns)

    ## store original data types
    feat_dtypes_orig = [data.iloc[:, j].dtype for j in range(d)]

    original_index = data.index.copy()

    # antes de mover y, extraio X e y originais para calcular IH
    X_for_ih = data.drop(columns=[y]).copy()
    y_for_ih = data[y].copy()

    # determinar coluna y e mover para a última posição (mantendo fluxo original)
    y_col = data.columns.get_loc(y)
    if y_col < d - 1:
        cols = list(range(d))
        cols[y_col], cols[d - 1] = cols[d - 1], cols[y_col]
        data = data[data.columns[cols]]

    # continuar com a codificação de colunas por índice
    feat_names = list(data.columns)
    data.columns = range(d)

    # criar y_sorted (séria com índices do dataframe original)
    y_df = pd.DataFrame(data[d - 1])
    y_sort = y_df.sort_values(by=d - 1)
    y_sort_series = y_sort[d - 1]  # Series ordenada com índices originais

    # -----------------------------
    # calcular IH usando a função fornecida (no dataset original)
    # -----------------------------
    # transformar X_for_ih e y_for_ih para numpy
    X_np = X_for_ih.to_numpy()
    y_np = y_for_ih.to_numpy().reshape(-1)

    ih_original = instance_hardness_regression(
        X_np,
        y_np
    )

    # criar Series de IH indexada pelo índice original do dataframe
    ih_series = pd.Series(ih_original, index=original_index)

    # reordenar ih para acompanhar y_sort (usando os índices ordenados de y_sort)
    y_ih = ih_series.loc[y_sort_series.index].values

    # validações análogas às do phi
    if all(i == 0 for i in y_ih):
        raise ValueError("redefine IH relevance function: all points are 0")

    if all(i == 1 for i in y_ih):
        raise ValueError("redefine IH relevance function: all points are 1")

    # ---------------------------------------------------------
    # determinar os bumps (raros vs normais) usando top 20% IH
    # ---------------------------------------------------------
    bumps = [0]
    # threshold dinâmico: percentil 80 (top 20% IH)
    dynamic_thres = np.percentile(y_ih, 70)

    for i in range(len(y_ih) - 1):
        if ((y_ih[i] >= dynamic_thres and y_ih[i + 1] < dynamic_thres) or
            (y_ih[i] < dynamic_thres and y_ih[i + 1] >= dynamic_thres)):
            bumps.append(i + 1)

    bumps.append(n)

    n_bumps = len(bumps) - 1

    ## determine indicies for each bump classification
    b_index = {}

    for i in range(n_bumps):
        b_index.update({i: y_sort[bumps[i]:bumps[i + 1]]})

    ## calculate over / under sampling percentage according to
    ## bump class and user specified method ("balance" or "extreme")
    b = round(n / n_bumps)
    s_perc = []
    scale = []
    obj = []

    if samp_method == "balance":
        for i in b_index:
            s_perc.append(b / len(b_index[i]))

    if samp_method == "extreme":
        for i in b_index:
            scale.append(b ** 2 / len(b_index[i]))
        scale = n_bumps * b / sum(scale)

        for i in b_index:
            obj.append(round(b ** 2 / len(b_index[i]) * scale, 2))
            s_perc.append(round(obj[i] / len(b_index[i]), 1))

    ## conduct over / under sampling and store modified training set
    data_new = pd.DataFrame()

    for i in range(n_bumps):

        ## no sampling
        if s_perc[i] == 1:

            ## simply return no sampling
            ## results to modified training set
            data_new = pd.concat([data.iloc[b_index[i].index], data_new], ignore_index = True)

        ## over-sampling
        if s_perc[i] > 1:

            ## generate synthetic observations in training set
            ## considered 'minority'
            ## (see 'over_sampling_gn()' function for details)
            synth_obs = over_sampling_gn(
                data = data,
                index = list(b_index[i].index),
                perc = s_perc[i] if not manual_perc else perc_o + 1,  # modified
                pert = pert,
                replace = replace  # added
            )

            ## concatenate over-sampling
            ## results to modified training set
            data_new = pd.concat([synth_obs, data_new], ignore_index = True)

            # added
            ## concatenate original data
            ## to modified training set
            original_obs = data.iloc[list(b_index[i].index)]
            data_new = pd.concat([original_obs, data_new], ignore_index = True)

        ## under-sampling
        if s_perc[i] < 1:  # exchanged

            if under_samp is True:  # exchanged

                ## drop observations in training set
                ## considered 'normal' (not 'rare')
                chosen_index = np.random.choice(  # modified
                    a = list(b_index[i].index),
                    size = int((s_perc[i] if not manual_perc else perc_u) * len(b_index[i])),
                    replace = replace
                )

                chosen_obs = data.iloc[chosen_index]  # modified

                ## concatenate under-sampling
                ## results to modified training set
                data_new = pd.concat([chosen_obs, data_new], ignore_index = True)  # modified

            # added
            ## concatenate 'normal' data
            ## to modified training set
            ## without undersamping
            else:
                original_obs = data.iloc[list(b_index[i].index)]
                data_new = pd.concat([original_obs, data_new], ignore_index = True)


    ## rename feature headers to originals
    data_new.columns = feat_names

    ## restore response variable y to original position
    if y_col < d - 1:
        cols = list(range(d))
        cols[y_col], cols[d - 1] = cols[d - 1], cols[y_col]
        data_new = data_new[data_new.columns[cols]]

    ## restore original data types
    for j in range(d):
        data_new.iloc[:, j] = data_new.iloc[:, j].astype(feat_dtypes_orig[j])

    ## return modified training set
    return data_new

In [8]:
# def hardness_bin_oversampling(df, target_col, threshold=0.8, bins=60):
#     """
#     Aplica RO apenas nos bins onde a média do IH está entre os top 20%.
#     """
#     y = df[target_col].values
#     X = df.drop(columns=[target_col]).values

#     # Instance Hardness
#     ih_scores = instance_hardness_regression(X, y)

#     # Threshold global (top 20% mais difíceis)
#     global_threshold = np.quantile(ih_scores, threshold)

#     # Gerar bins
#     counts, bin_edges = np.histogram(y, bins=bins)

#     new_df = df.copy()

#     for i in range(len(bin_edges)-1):
#         left = bin_edges[i]
#         right = bin_edges[i+1]

#         mask = (y >= left) & (y < right)
#         bin_data = df[mask]

#         if len(bin_data) == 0:
#             continue

#         ih_bin = ih_scores[mask]
#         mean_ih = np.mean(ih_bin)

#         # Se bin estiver entre os mais difíceis → aplica oversampling
#         if mean_ih >= global_threshold and len(bin_data) > 5:
#             sampled = bin_data.sample(
#                 n=len(bin_data),
#                 replace=True,
#                 random_state=42
#             )
#             new_df = pd.concat([new_df, sampled], ignore_index=True)

#     return new_df

In [9]:
def apply_iblr_ro(data, target, method="balance", rel_thres=0.8):
    """
    Wrapper do IBLR-RO

    data: DataFrame completo (X + y)
    target: nome da coluna target
    method: método de amostragem (ex: "balance", "random"...)
    rel_thres: limiar de relevância
    """
    try:
        data = data.copy()
        data.columns = data.columns.astype(str)  # garante compatibilidade

        new_data = iblr.ro(
            data=data,
            y=str(target),
            samp_method=method,
            rel_thres=rel_thres
        )
        return new_data

    except Exception as e:
        print("⚠️ IBLR-RO falhou:", e)
        return data.copy()


def apply_iblr_gn(data, target, method="balance", pert=0.1, rel_thres=0.8):
    """
    Wrapper do IBLR-GN

    data: DataFrame completo (X + y)
    target: nome da coluna target
    method: método de amostragem (ex: "balance", "random"...)
    pert: nível de perturbação
    rel_thres: limiar de relevância
    """
    try:
        data = data.copy()
        data.columns = data.columns.astype(str)  # garante compatibilidade

        new_data = iblr.gn(
            data=data,
            y=str(target),
            samp_method=method,
            pert=pert,
            rel_thres=rel_thres
        )
        return new_data

    except Exception as e:
        print("⚠️ IBLR-GN falhou:", e)
        return data.copy()



In [10]:

def evaluate_model(X_train, X_test, y_train, y_test, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # MSE
    mse = mean_squared_error(y_test, y_pred)

    # mae
    mae = mean_absolute_error(y_test, y_pred)

    return mse, mae


In [11]:
models = {
    "SVR": SVR(),  # não possui random_state
    "NNET":MLPRegressor(hidden_layer_sizes=(50,50), max_iter=1000, random_state=42),
    "XGB":XGBRegressor(n_estimators=100, random_state=42),
    "RF":  RandomForestRegressor(n_estimators=100, random_state=42),
    "BAGGING": BaggingRegressor(random_state=42)
}

In [12]:
wine = pd.read_csv('/content/drive/MyDrive/TCC/dados/wine_quality.csv')
a1 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a1.csv')
a2 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a2.csv')
a3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a3.csv')
a7 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a7.csv')
acceleration = pd.read_csv('/content/drive/MyDrive/TCC/dados/acceleration.csv')
airfoild = pd.read_csv('/content/drive/MyDrive/TCC/dados/airfoild.csv')
boston = pd.read_csv('/content/drive/MyDrive/TCC/dados/boston.csv')
concreteStrength = pd.read_csv('/content/drive/MyDrive/TCC/dados/concreteStrength.csv')
heat = pd.read_csv('/content/drive/MyDrive/TCC/dados/heat.csv')
sensory = pd.read_csv('/content/drive/MyDrive/TCC/dados/sensory.csv')
analcatdata_apnea3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/analcatdata_apnea3.csv')
available_power = pd.read_csv('/content/drive/MyDrive/TCC/dados/available_power.csv')
cocomo_numeric = pd.read_csv('/content/drive/MyDrive/TCC/dados/cocomo_numeric.csv')
debutanizer = pd.read_csv('/content/drive/MyDrive/TCC/dados/debutanizer.csv')
fuel_consumption_country = pd.read_csv('/content/drive/MyDrive/TCC/dados/fuel_consumption_country.csv')

abalone = pd.read_csv('/content/drive/MyDrive/TCC/dados/abalone.csv')
california = pd.read_csv('/content/drive/MyDrive/TCC/dados/california.csv')
compactiv = pd.read_csv('/content/drive/MyDrive/TCC/dados/compactiv.csv')
cpu_small = pd.read_csv('/content/drive/MyDrive/TCC/dados/cpu_small.csv')
forestFires = pd.read_csv('/content/drive/MyDrive/TCC/dados/forestFires.csv')
kdd_coil_1 = pd.read_csv('/content/drive/MyDrive/TCC/dados/kdd_coil_1.csv')
lungcancer_shedden = pd.read_csv('/content/drive/MyDrive/TCC/dados/lungcancer_shedden.csv')
maximal_torque = pd.read_csv('/content/drive/MyDrive/TCC/dados/maximal_torque.csv')
meta = pd.read_csv('/content/drive/MyDrive/TCC/dados/meta.csv')
mortgage = pd.read_csv('/content/drive/MyDrive/TCC/dados/mortgage.csv')
pdgfr = pd.read_csv('/content/drive/MyDrive/TCC/dados/pdgfr.csv')
space_ga = pd.read_csv('/content/drive/MyDrive/TCC/dados/space_ga.csv')
treasury = pd.read_csv('/content/drive/MyDrive/TCC/dados/treasury.csv')
triazines = pd.read_csv('/content/drive/MyDrive/TCC/dados/triazines.csv')



datasets = {
    "a1": {"data": a1, "target": "a1"},
     "a2": {"data": a2, "target": "a2"},
     "a3": {"data": a3, "target": "a3"},
     "a7": {"data": a7, "target": "a7"},
      "abalone": {"data": abalone, "target": "Rings"},
      "acceleration": {"data": acceleration, "target": "target"},
     "airfoild": {"data": airfoild, "target": "target"},
     "analcatdata_apnea3": {"data": analcatdata_apnea3, "target": "target"},
      "available_power": {"data": available_power, "target": "target"},
   "boston": {"data": boston, "target": "target"},
    #"california": {"data": california, "target": "target"},
    "cocomo_numeric": {"data": cocomo_numeric, "target": "target"},
    "compactiv": {"data": compactiv, "target": "target"},
     "concreteStrength": {"data": concreteStrength, "target": "target"},
     "cpu_small": {"data": cpu_small, "target": "usr"},
     "debutanizer": {"data": debutanizer, "target": "target"},
    "fuel_consumption_country": {"data": fuel_consumption_country, "target": "fuel.consumption.country"},
    "forestFires": {"data": forestFires, "target": "Area"},
     "heat": {"data": heat, "target": "heat"},
     "kdd_coil_1": {"data": kdd_coil_1, "target": "target"},
     "lungcancer_shedden": {"data": lungcancer_shedden, "target": "target"},
      "maximal_torque": {"data": maximal_torque, "target": "maximal.torque"},
    "meta": {"data": meta, "target": "target"},
    "mortgage": {"data": mortgage, "target": "target"},
    "pdgfr": {"data": pdgfr, "target": "target"},
    "sensory": {"data": sensory, "target": "target"},
     "space_ga": {"data": space_ga, "target": "target"},
    "treasury": {"data": treasury, "target": "target"},
    "triazines": {"data": triazines, "target": "target"},
    "wine": {"data": wine, "target": "target"},
}


Teste para GN

In [13]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

results = []

for name, info in datasets.items():
    df = info["data"].copy()
    target = info["target"]

    print(f"\nProcessando: {name}")

    X = df.drop(columns=[target])
    y = df[target]

    # Loop KFold
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        print(f"  Fold {fold+1}/5")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        base_train = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)

        # Agora o oversampling acontece AQUI, dentro do fold
        try:
            train_smogn = apply_iblr_gn(base_train, target)
        except Exception as e:
            print(f"⚠️ RO falhou no fold {fold+1}: {e}")
            train_smogn = base_train.copy()

        train_hard = apply_gn_ih(base_train, target)

        versions = {
            "Original": base_train,
            "RO": train_smogn,
            "Hardness-RO": train_hard
        }

        for model_name, model in models.items():
            for version_name, train_data in versions.items():

                Xtr = train_data.drop(columns=[target])
                ytr = train_data[target]

                Xts = X_test
                yts = y_test

                mse, mae = evaluate_model(
                    Xtr, Xts, ytr, yts, model
                )

                results.append({
                    "Dataset": name,
                    "Fold": fold + 1,
                    "Model": model_name,
                    "Versao": version_name,
                    "MSE": mse,
                    "MAE": mae
                })


Output hidden; open in https://colab.research.google.com to view.

In [14]:
results_df = pd.DataFrame(results)
results_df.head(50)


,Dataset,Fold,Model,Versao,MSE,MAE
0,a1,1,SVR,Original,5.048297e+02,14.816346
1,a1,1,SVR,RO,7.776990e+02,24.496299
2,a1,1,SVR,Hardness-RO,6.619968e+02,16.435538
3,a1,1,NNET,Original,8.927546e+05,311.047890
4,a1,1,NNET,RO,1.926473e+06,515.547667
5,a1,1,NNET,Hardness-RO,2.247415e+06,554.354393
6,a1,1,XGB,Original,3.554850e+02,14.716183
7,a1,1,XGB,RO,4.976124e+02,17.342458
8,a1,1,XGB,Hardness-RO,6.268892e+02,16.302279
9,a1,1,RF,Original,2.913354e+02,13.882875


In [15]:
results_df.to_csv("comparacao_final_GN.csv", index=False)

In [16]:
# Definir a ordem desejada
ordem_versoes = ["Original", "RO", "Hardness-RO"]

results_df["Versao"] = pd.Categorical(
    results_df["Versao"],
    categories=ordem_versoes,
    ordered=True
)

mse_por_modelo = (
    results_df
    .groupby(["Dataset", "Model", "Versao"], as_index=False)
    .agg(
        MSE_medio=("MSE", "mean"),
        MAE_medio=("MAE", "mean")
        )
    .sort_values(["Dataset", "Model", "Versao"])
)

mse_por_modelo.head(60)

/tmp/ipython-input-4070163773.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Dataset", "Model", "Versao"], as_index=False)


,Dataset,Model,Versao,MSE_medio,MAE_medio
0,a1,BAGGING,Original,3.132490e+02,12.003677
1,a1,BAGGING,RO,4.407671e+02,15.691707
2,a1,BAGGING,Hardness-RO,3.403349e+02,12.619130
3,a1,NNET,Original,4.597342e+05,225.520656
4,a1,NNET,RO,1.557393e+06,409.248418
5,a1,NNET,Hardness-RO,6.295769e+05,244.515487
6,a1,RF,Original,2.720753e+02,11.520841
7,a1,RF,RO,3.755301e+02,14.652916
8,a1,RF,Hardness-RO,3.182442e+02,12.277959
9,a1,SVR,Original,4.792085e+02,14.404502


### Vendo quem foi melhor pro MSE

In [17]:
import pandas as pd

# ===============================
# 1. Ordenar versões corretamente
# ===============================

ordem_versoes = ["Original", "RO", "Hardness-RO"]

results_df["Versao"] = pd.Categorical(
    results_df["Versao"],
    categories=ordem_versoes,
    ordered=True
)

# ===============================
# 2. Agregar MSE médio por fold
# ===============================

mse_por_modelo = (
    results_df
    .groupby(["Dataset", "Model", "Versao"], as_index=False)
    .agg(MSE_medio=("MSE", "mean"))
    .sort_values(["Dataset", "Model", "Versao"])
)

print("\n📊 MSE médio por versão:")
print(mse_por_modelo)

# ===============================
# 3. Comparação entre RO e Hardness
# ===============================

# Filtrar somente RO e Hardness-RO
comp = mse_por_modelo[
    mse_por_modelo["Versao"].isin(["RO", "Hardness-RO"])
]

# Pivot: transformar em colunas RO e Hardness-RO
pivot = comp.pivot(
    index=["Dataset", "Model"],
    columns="Versao",
    values="MSE_medio"
).reset_index()

print("\n📌 Comparação direta:")
print(pivot)

# ===============================
# 4. Quem foi melhor?
# ===============================

pivot["Hardness_melhor_que_RO"] = pivot["Hardness-RO"] < pivot["RO"]
pivot["RO_melhor_que_Hardness"] = pivot["RO"] < pivot["Hardness-RO"]
pivot["Empate"] = pivot["RO"] == pivot["Hardness-RO"]

# ===============================
# 5. Contagem geral
# ===============================

total_hardness = pivot["Hardness_melhor_que_RO"].sum()
total_ro = pivot["RO_melhor_que_Hardness"].sum()
empates = pivot["Empate"].sum()

print(f"\n🏆 RESULTADO GERAL:")
print(f"Hardness-RO foi melhor em {total_hardness} pares (dataset, modelo)")
print(f"RO foi melhor em {total_ro} pares (dataset, modelo)")
print(f"Houve {empates} empates")

# ===============================
# 6. Contagem por dataset
# ===============================

hardness_por_dataset = (
    pivot[pivot["Hardness_melhor_que_RO"]]
    .groupby("Dataset")["Model"]
    .nunique()
    .reset_index(name="Modelos_ganhos_Hardness")
)

ro_por_dataset = (
    pivot[pivot["RO_melhor_que_Hardness"]]
    .groupby("Dataset")["Model"]
    .nunique()
    .reset_index(name="Modelos_ganhos_RO")
)

print("\n📌 Hardness-RO melhor que RO por dataset:")
print(hardness_por_dataset)

print("\n📌 RO melhor que Hardness-RO por dataset:")
print(ro_por_dataset)

# ===============================
# 7. Frases finais (estilo artigo)
# ===============================

datasets_hardness = hardness_por_dataset["Dataset"].nunique()
datasets_ro = ro_por_dataset["Dataset"].nunique()

modelos_hardness = hardness_por_dataset["Modelos_ganhos_Hardness"]



📊 MSE médio por versão:
    Dataset    Model       Versao     MSE_medio
0        a1  BAGGING     Original  3.132490e+02
1        a1  BAGGING           RO  4.407671e+02
2        a1  BAGGING  Hardness-RO  3.403349e+02
3        a1     NNET     Original  4.597342e+05
4        a1     NNET           RO  1.557393e+06
..      ...      ...          ...           ...
430    wine      SVR           RO  8.905726e-01
431    wine      SVR  Hardness-RO  6.784585e-01
432    wine      XGB     Original  4.022404e-01
433    wine      XGB           RO  5.222591e-01
434    wine      XGB  Hardness-RO  4.776108e-01

[435 rows x 4 columns]

📌 Comparação direta:
Versao Dataset    Model            RO    Hardness-RO
0           a1  BAGGING  4.407671e+02     340.334936
1           a1     NNET  1.557393e+06  629576.854848
2           a1       RF  3.755301e+02     318.244176
3           a1      SVR  7.767297e+02     533.206426
4           a1      XGB  4.317491e+02     390.264214
..         ...      ...           .

/tmp/ipython-input-2671954749.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Dataset", "Model", "Versao"], as_index=False)


In [18]:
# Separar RO e Hardness-RO
ro_df = mse_por_modelo[mse_por_modelo["Versao"] == "RO"]
hard_df = mse_por_modelo[mse_por_modelo["Versao"] == "Hardness-RO"]

# Juntar lado a lado
comparacao = pd.merge(
    ro_df,
    hard_df,
    on=["Dataset", "Model"],
    suffixes=("_RO", "_HardnessRO")
)

# Criar colunas indicando quem foi melhor (menor MSE)
comparacao["Hardness_melhor"] = comparacao["MSE_medio_HardnessRO"] < comparacao["MSE_medio_RO"]
comparacao["RO_melhor"] = comparacao["MSE_medio_RO"] < comparacao["MSE_medio_HardnessRO"]
comparacao["Empate"] = comparacao["MSE_medio_RO"] == comparacao["MSE_medio_HardnessRO"]

datasets_ro_ganhou = (
    comparacao[comparacao["RO_melhor"]]
    .groupby("Dataset")["Model"]
    .apply(list)
    .reset_index(name="Modelos_que_RO_ganhou")
)

datasets_ro_ganhou



,Dataset,Modelos_que_RO_ganhou
0,a7,[NNET]
1,acceleration,"[BAGGING, NNET, RF, XGB]"
2,airfoild,"[BAGGING, RF, SVR, XGB]"
3,analcatdata_apnea3,"[BAGGING, NNET, RF, SVR, XGB]"
4,available_power,"[BAGGING, RF, SVR, XGB]"
5,boston,"[BAGGING, RF, XGB]"
6,cocomo_numeric,"[BAGGING, NNET, RF, SVR, XGB]"
7,compactiv,"[RF, SVR, XGB]"
8,concreteStrength,"[BAGGING, RF]"
9,cpu_small,[XGB]


In [19]:
datasets_hard_ganhou = (
    comparacao[comparacao["Hardness_melhor"]]
    .groupby("Dataset")["Model"]
    .apply(list)
    .reset_index(name="Modelos_que_HardnessRO_ganhou")
)

datasets_hard_ganhou


,Dataset,Modelos_que_HardnessRO_ganhou
0,a1,"[BAGGING, NNET, RF, SVR, XGB]"
1,a2,"[BAGGING, NNET, RF, SVR, XGB]"
2,a3,"[BAGGING, NNET, RF, SVR, XGB]"
3,a7,"[BAGGING, RF, SVR, XGB]"
4,abalone,"[BAGGING, NNET, RF, SVR, XGB]"
5,acceleration,[SVR]
6,airfoild,[NNET]
7,available_power,[NNET]
8,boston,"[NNET, SVR]"
9,compactiv,"[BAGGING, NNET]"


In [20]:
# Agrupar por modelo e contar vitórias
comparacao_por_modelo = (
    comparacao
    .groupby("Model")
    .agg(
        Vitorias_RO=("RO_melhor", "sum"),
        Vitorias_HardnessRO=("Hardness_melhor", "sum"),
        Empates=("Empate", "sum"),
        Total_Comparacoes=("Model", "count")
    )
    .reset_index()
)

# Criar também a diferença de vitórias
comparacao_por_modelo["Dif_Hardness_minus_RO"] = (
    comparacao_por_modelo["Vitorias_HardnessRO"]
    - comparacao_por_modelo["Vitorias_RO"]
)

comparacao_por_modelo.sort_values("Dif_Hardness_minus_RO", ascending=False)


,Model,Vitorias_RO,Vitorias_HardnessRO,Empates,Total_Comparacoes,Dif_Hardness_minus_RO
3,SVR,8,21,0,29,13
1,NNET,11,18,0,29,7
0,BAGGING,12,17,0,29,5
4,XGB,14,15,0,29,1
2,RF,16,13,0,29,-3


### Vendo quem foi melhor pro mae

In [21]:
import pandas as pd

# ===============================
# 1. Ordenar versões corretamente
# ===============================

ordem_versoes = ["Original", "RO", "Hardness-RO"]

results_df["Versao"] = pd.Categorical(
    results_df["Versao"],
    categories=ordem_versoes,
    ordered=True
)

# ===============================
# 2. Agregar mae médio por fold
# ===============================

mae_por_modelo = (
    results_df
    .groupby(["Dataset", "Model", "Versao"], as_index=False)
    .agg(mae_medio=("MAE", "mean"))
    .sort_values(["Dataset", "Model", "Versao"])
)

print("\n📊 mae médio por versão:")
print(mae_por_modelo)

# ===============================
# 3. Comparação entre RO e Hardness
# ===============================

# Filtrar apenas RO e Hardness-RO
comp = mae_por_modelo[
    mae_por_modelo["Versao"].isin(["RO", "Hardness-RO"])
]

# Pivot para comparar lado a lado
pivot = comp.pivot(
    index=["Dataset", "Model"],
    columns="Versao",
    values="mae_medio"
).reset_index()

print("\n📌 Comparação direta (mae):")
print(pivot)

# ===============================
# 4. Quem foi melhor?
# ===============================

pivot["Hardness_melhor_que_RO"] = pivot["Hardness-RO"] < pivot["RO"]
pivot["RO_melhor_que_Hardness"] = pivot["RO"] < pivot["Hardness-RO"]
pivot["Empate"] = pivot["RO"] == pivot["Hardness-RO"]

# ===============================
# 5. Contagem geral
# ===============================

total_hardness = pivot["Hardness_melhor_que_RO"].sum()
total_ro = pivot["RO_melhor_que_Hardness"].sum()
empates = pivot["Empate"].sum()

print(f"\n🏆 RESULTADO GERAL (mae):")
print(f"Hardness-RO foi melhor em {total_hardness} pares (dataset, modelo)")
print(f"RO foi melhor em {total_ro} pares (dataset, modelo)")
print(f"Houve {empates} empates")

# ===============================
# 6. Contagem por dataset
# ===============================

hardness_por_dataset = (
    pivot[pivot["Hardness_melhor_que_RO"]]
    .groupby("Dataset")["Model"]
    .nunique()
    .reset_index(name="Modelos_ganhos_Hardness")
)

ro_por_dataset = (
    pivot[pivot["RO_melhor_que_Hardness"]]
    .groupby("Dataset")["Model"]
    .nunique()
    .reset_index(name="Modelos_ganhos_RO")
)

print("\n📌 Hardness-RO melhor que RO por dataset (mae):")
print(hardness_por_dataset)

print("\n📌 RO melhor que Hardness-RO por dataset (mae):")
print(ro_por_dataset)

# ===============================
# 7. Frases finais (estilo artigo)
# ===============================

datasets_hardness = hardness_por_dataset["Dataset"].nunique()
datasets_ro = ro_por_dataset["Dataset"].nunique()

modelos_hardness = hardness_por_dataset["Modelos_ganhos_Hardness"].sum()
modelos_ro = ro_por_dataset["Modelos_ganhos_RO"].sum()

print("\n📄 Resumo para artigo (mae):")
print(
    f"O método Hardness-RO apresentou melhor desempenho em {datasets_hardness} datasets, "
    f"superando RO em {modelos_hardness} combinações de modelos. "
    f"Por outro lado, o método RO obteve melhor desempenho em {datasets_ro} datasets, "
    f"vencendo em {modelos_ro} combinações."
)


/tmp/ipython-input-1476718455.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["Dataset", "Model", "Versao"], as_index=False)



📊 mae médio por versão:
    Dataset    Model       Versao   mae_medio
0        a1  BAGGING     Original   12.003677
1        a1  BAGGING           RO   15.691707
2        a1  BAGGING  Hardness-RO   12.619130
3        a1     NNET     Original  225.520656
4        a1     NNET           RO  409.248418
..      ...      ...          ...         ...
430    wine      SVR           RO    0.741672
431    wine      SVR  Hardness-RO    0.636015
432    wine      XGB     Original    0.455055
433    wine      XGB           RO    0.541742
434    wine      XGB  Hardness-RO    0.501772

[435 rows x 4 columns]

📌 Comparação direta (mae):
Versao Dataset    Model          RO  Hardness-RO
0           a1  BAGGING   15.691707    12.619130
1           a1     NNET  409.248418   244.515487
2           a1       RF   14.652916    12.277959
3           a1      SVR   24.705988    14.938645
4           a1      XGB   14.729906    13.100458
..         ...      ...         ...          ...
140       wine  BAGGING    0

In [22]:
# Separar RO e Hardness-RO usando mae_medio
ro_df = mae_por_modelo[mae_por_modelo["Versao"] == "RO"]
hard_df = mae_por_modelo[mae_por_modelo["Versao"] == "Hardness-RO"]

# Juntar lado a lado
comparacao_mae = pd.merge(
    ro_df,
    hard_df,
    on=["Dataset", "Model"],
    suffixes=("_RO", "_HardnessRO")
)

# Criar colunas indicando quem foi melhor (menor mae)
comparacao_mae["Hardness_melhor"] = (
    comparacao_mae["mae_medio_HardnessRO"] < comparacao_mae["mae_medio_RO"]
)

comparacao_mae["RO_melhor"] = (
    comparacao_mae["mae_medio_RO"] < comparacao_mae["mae_medio_HardnessRO"]
)

comparacao_mae["Empate"] = (
    comparacao_mae["mae_medio_RO"] == comparacao_mae["mae_medio_HardnessRO"]
)

# Listar os modelos em que RO ganhou por dataset
datasets_ro_ganhou_phi = (
    comparacao_mae[comparacao_mae["RO_melhor"]]
    .groupby("Dataset")["Model"]
    .apply(list)
    .reset_index(name="Modelos_que_RO_ganhou")
)

datasets_ro_ganhou_phi


,Dataset,Modelos_que_RO_ganhou
0,a7,[NNET]
1,acceleration,[NNET]
2,airfoild,"[BAGGING, RF, XGB]"
3,analcatdata_apnea3,"[BAGGING, NNET, RF, XGB]"
4,available_power,"[BAGGING, RF, SVR, XGB]"
5,boston,[BAGGING]
6,cocomo_numeric,"[BAGGING, NNET, RF, XGB]"
7,concreteStrength,[RF]
8,cpu_small,[NNET]
9,heat,"[BAGGING, RF, XGB]"


In [23]:
datasets_hard_ganhou_mae = (
    comparacao_mae[comparacao_mae["Hardness_melhor"]]
    .groupby("Dataset")["Model"]
    .apply(list)
    .reset_index(name="Modelos_que_HardnessRO_ganhou")
)

datasets_hard_ganhou_mae


,Dataset,Modelos_que_HardnessRO_ganhou
0,a1,"[BAGGING, NNET, RF, SVR, XGB]"
1,a2,"[BAGGING, NNET, RF, SVR, XGB]"
2,a3,"[BAGGING, NNET, RF, SVR, XGB]"
3,a7,"[BAGGING, RF, SVR, XGB]"
4,abalone,"[BAGGING, NNET, RF, SVR, XGB]"
5,acceleration,"[BAGGING, RF, SVR, XGB]"
6,airfoild,"[NNET, SVR]"
7,analcatdata_apnea3,[SVR]
8,available_power,[NNET]
9,boston,"[NNET, RF, SVR, XGB]"


In [24]:
# Agrupar por modelo e contar vitórias (mae)
comparacao_por_modelo_mae = (
    comparacao_mae
    .groupby("Model")
    .agg(
        Vitorias_RO=("RO_melhor", "sum"),
        Vitorias_HardnessRO=("Hardness_melhor", "sum"),
        Empates=("Empate", "sum"),
        Total_Comparacoes=("Model", "count")
    )
    .reset_index()
)

# Criar também a diferença de vitórias
comparacao_por_modelo_mae["Dif_Hardness_minus_RO"] = (
    comparacao_por_modelo_mae["Vitorias_HardnessRO"]
    - comparacao_por_modelo_mae["Vitorias_RO"]
)

# Ordenar pela diferença (quem mais ganhou)
comparacao_por_modelo_mae.sort_values("Dif_Hardness_minus_RO", ascending=False)


,Model,Vitorias_RO,Vitorias_HardnessRO,Empates,Total_Comparacoes,Dif_Hardness_minus_RO
3,SVR,2,27,0,29,25
0,BAGGING,9,20,0,29,11
1,NNET,9,20,0,29,11
4,XGB,9,20,0,29,11
2,RF,10,19,0,29,9


## Erro por area

In [25]:
def plot_binned_mae(y_true, y_pred, n_bins=10, title="Binned MAE"):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    abs_error = np.abs(y_true - y_pred)

    # Criar bins
    bins = np.linspace(y_true.min(), y_true.max(), n_bins + 1)

    # Atribuir cada valor a um bin
    bin_ids = np.digitize(y_true, bins) - 1

    binned_mae = []
    bin_centers = []

    for i in range(n_bins):
        mask = bin_ids == i

        if np.sum(mask) == 0:
            binned_mae.append(np.nan)
        else:
            binned_mae.append(np.mean(abs_error[mask]))

        bin_centers.append((bins[i] + bins[i+1]) / 2)

    # Plot
    fig, ax1 = plt.subplots(figsize=(9,5))

    # Histograma
    ax1.hist(y_true, bins=bins, color='indianred', alpha=0.6)
    ax1.set_ylabel("Count", color='indianred')
    ax1.set_xlabel("Target")
    ax1.set_title(title)

    # Segundo eixo para MAE
    ax2 = ax1.twinx()
    ax2.plot(bin_centers, binned_mae, marker='o')
    ax2.set_ylabel("Binned MAE")

    plt.show()

In [26]:

from sklearn.model_selection import train_test_split

for name, info in datasets.items():
    df = info["data"].copy()
    target = info["target"]

    print(f"\n📊 Dataset: {name}")

    X = df.drop(columns=[target])
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    train_original = pd.concat([X_train, y_train], axis=1)

    for model_name, model in models.items():
        print(f"  🔷 Modelo: {model_name} (Original)")

        Xtr = train_original.drop(columns=[target])
        ytr = train_original[target]

        model.fit(Xtr, ytr)

        y_pred = model.predict(X_test)

        plot_binned_mae(
            y_true=y_test,
            y_pred=y_pred,
            n_bins=15,
            title=f"{name} - {model_name} - Original"
        )


Output hidden; open in https://colab.research.google.com to view.